In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")


Project root added: C:\Users\Radimon\2025_japan_people_flow
✓ Imports ready


In [2]:
import joblib
model = joblib.load("../models/lgbm_sapporo.pkl")

features = [
    "weekday", "t", "x", "y", "is_weekend",
    "lag_1", "lag_7", "rolling_3", "rolling_7"
]

print("✓ Model loaded")


✓ Model loaded


In [3]:
df = pd.read_parquet("../data/processed/sapporo_density.parquet")

# d=0 假設是 2023-01-01（你之後如果拿到真實起始日再換）
df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

print("Rows:", len(df))
print("d range:", df["d"].min(), df["d"].max())
print("unique days:", df["d"].nunique())
df.head()


Rows: 6024808
d range: 0 74
unique days: 75


,d,t,x,y,count,date,weekday,is_weekend
0,0,0,23,153,23,2023-01-01,6,1
1,0,0,24,153,20,2023-01-01,6,1
2,0,0,17,163,8,2023-01-01,6,1
3,0,0,18,156,7,2023-01-01,6,1
4,0,0,22,151,7,2023-01-01,6,1


In [4]:
df = df.sort_values(["x","y","t","d"])

df["lag_1"] = df.groupby(["x","y","t"])["count"].shift(1)
df["lag_7"] = df.groupby(["x","y","t"])["count"].shift(7)

df["rolling_3"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["rolling_7"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df_feat = df.dropna().copy()
print("after feature rows:", len(df_feat))
df_feat.head()


after feature rows: 4881953


,d,t,x,y,count,date,weekday,is_weekend,lag_1,lag_7,rolling_3,rolling_7
5061512,62,13,1,24,1,2023-03-04,5,1,1.0,1.0,1.333333,1.142857
5513835,68,13,1,24,1,2023-03-10,4,0,1.0,1.0,1.000000,1.142857
3707330,45,14,1,24,1,2023-02-15,2,0,1.0,1.0,1.333333,1.142857
4264450,52,14,1,24,1,2023-02-22,2,0,1.0,1.0,1.000000,1.142857
4345854,53,14,1,24,1,2023-02-23,3,0,1.0,1.0,1.000000,1.142857


In [5]:
def make_query_frame(df_feat: pd.DataFrame, target_weekday: int, target_t: int) -> pd.DataFrame:
    """
    用最近 7 天同 (x,y,t) 的 lag/rolling 平均當作 proxy
    產生下一次查詢可用的 feature frame
    """
    max_day = df_feat["d"].max()
    recent = df_feat[df_feat["d"] > max_day - 7].copy()

    base = (recent[recent["t"] == target_t]
            .groupby(["x","y"], as_index=False)
            .agg({
                "lag_1": "mean",
                "lag_7": "mean",
                "rolling_3": "mean",
                "rolling_7": "mean",
            }))

    base["weekday"] = target_weekday
    base["t"] = target_t
    base["is_weekend"] = int(target_weekday >= 5)

    return base


In [6]:
def predict_with_levels(model, qdf: pd.DataFrame, q_high=0.95, q_med=0.80, q_low=0.60):
    qdf = qdf.copy()
    qdf["pred_count"] = model.predict(qdf[features])

    thr_high = qdf["pred_count"].quantile(q_high)
    thr_med  = qdf["pred_count"].quantile(q_med)
    thr_low  = qdf["pred_count"].quantile(q_low)

    def level(v):
        if v >= thr_high: return "HIGH"
        if v >= thr_med:  return "MED"
        if v >= thr_low:  return "LOW"
        return "NONE"

    qdf["level"] = qdf["pred_count"].apply(level)

    return qdf, {"HIGH": thr_high, "MED": thr_med, "LOW": thr_low}


In [7]:
# Friday=4, 18:00 => t=36
target_weekday = 4
target_t = 36

qdf = make_query_frame(df_feat, target_weekday, target_t)
out, thresholds = predict_with_levels(model, qdf)

print("thresholds:", thresholds)
print(out["level"].value_counts())

out.sort_values("pred_count", ascending=False).head(10)


thresholds: {'HIGH': np.float64(8.419883656186002), 'MED': np.float64(4.311187152998111), 'LOW': np.float64(2.5035314754999733)}
level
NONE    1565
LOW      521
MED      391
HIGH     131
Name: count, dtype: int64


,x,y,lag_1,lag_7,rolling_3,rolling_7,weekday,t,is_weekend,pred_count,level
2607,999,999,1015.000000,982.714286,1014.285714,1002.448980,4,36,0,636.188726,HIGH
900,25,153,81.714286,81.000000,82.857143,81.367347,4,36,0,101.569285,HIGH
951,26,153,95.571429,88.857143,93.238095,91.836735,4,36,0,87.159203,HIGH
852,24,153,67.857143,64.857143,65.761905,65.020408,4,36,0,77.642978,HIGH
950,26,152,60.142857,58.000000,60.047619,59.857143,4,36,0,62.876747,HIGH
899,25,152,33.285714,34.714286,34.238095,34.979592,4,36,0,40.485930,HIGH
850,24,151,25.714286,25.428571,26.238095,25.714286,4,36,0,30.764081,HIGH
901,25,154,28.000000,27.142857,26.952381,26.836735,4,36,0,29.762956,HIGH
851,24,152,24.571429,21.857143,24.142857,24.183673,4,36,0,28.233470,HIGH
683,20,177,23.428571,22.285714,22.761905,21.959184,4,36,0,27.073759,HIGH


In [8]:
from sklearn.cluster import DBSCAN

# 只對 HIGH 做聚類
high_df = out[out["level"] == "HIGH"].copy()

coords = high_df[["x", "y"]].values

# eps 要依 grid 尺寸調整
clusterer = DBSCAN(eps=1, min_samples=5)
high_df["cluster"] = clusterer.fit_predict(coords)

print("Clusters found:", high_df["cluster"].nunique())
high_df.head()


Clusters found: 2


,x,y,lag_1,lag_7,rolling_3,rolling_7,weekday,t,is_weekend,pred_count,level,cluster
211,8,176,9.285714,9.000000,9.190476,9.183673,4,36,0,9.541701,HIGH,-1
394,13,169,8.714286,11.428571,9.285714,10.163265,4,36,0,11.689483,HIGH,-1
436,14,169,7.285714,7.285714,7.904762,8.163265,4,36,0,8.703778,HIGH,-1
501,16,156,10.142857,10.857143,10.619048,10.816327,4,36,0,11.850159,HIGH,-1
505,16,160,9.142857,7.857143,8.714286,8.244898,4,36,0,8.512176,HIGH,-1


In [9]:
clusters = []

for cid in high_df["cluster"].unique():
    if cid == -1:
        continue  # 忽略雜訊點
    
    group = high_df[high_df["cluster"] == cid]
    
    center_x = group["x"].mean()
    center_y = group["y"].mean()
    
    # 半徑：取最遠距離
    radius = np.sqrt(
        ((group["x"] - center_x)**2 +
         (group["y"] - center_y)**2).max()
    )
    
    clusters.append({
        "cluster_id": cid,
        "center_x": center_x,
        "center_y": center_y,
        "radius": radius,
        "points": len(group)
    })

cluster_df = pd.DataFrame(clusters)
cluster_df.sort_values("points", ascending=False)


,cluster_id,center_x,center_y,radius,points
0,0,25.0,153.1875,3.941466,32


In [10]:
def build_clusters(df_level, eps):
    from sklearn.cluster import DBSCAN
    coords = df_level[["x","y"]].values
    db = DBSCAN(eps=eps, min_samples=5)
    df_level["cluster"] = db.fit_predict(coords)
    return df_level

med_df = build_clusters(out[out["level"]=="MED"].copy(), eps=2)
low_df = build_clusters(out[out["level"]=="LOW"].copy(), eps=3)


In [11]:
from src.geo.grid_to_latlng import GridLatLngMapper

anchors = [
    {"x": 24, "y": 151, "lat": 43.06918333153887, "lng": 141.35147072116592},  # 札幌站 
    {"x": 24, "y": 148, "lat": 43.07940372979633, "lng": 141.34225589803765},  # 北海道大學
    {"x": 26, "y": 153, "lat": 43.05798589528942, "lng": 141.35402112326315},  # 狸小路商店街
]

mapper = GridLatLngMapper(anchors)

cluster_geo = mapper.transform(cluster_df, x_col="center_x", y_col="center_y")

cluster_geo


,cluster_id,center_x,center_y,radius,points,lat,lng
0,0,25.0,153.1875,3.941466,32,43.059539,141.356393


In [12]:
# 假設你要預測：
# 2023-03-20 (星期一) 18:00

future_date = pd.to_datetime("2023-03-20")
future_weekday = future_date.weekday()
future_is_weekend = future_weekday >= 5
future_t = 36   # 18:00

# 取所有 grid 座標
grid_xy = df[["x","y"]].drop_duplicates()

future_df = grid_xy.copy()
future_df["weekday"] = future_weekday
future_df["is_weekend"] = future_is_weekend
future_df["t"] = future_t

# 🔥 這一步很重要：lag 特徵要來自歷史最後一天
latest_day = df["d"].max()

lag_features = df[df["d"] == latest_day][
    ["x","y","lag_1","lag_7","rolling_3","rolling_7"]
]

future_df = future_df.merge(
    lag_features,
    on=["x","y"],
    how="left"
)

# 預測
features = [
    "weekday","t","x","y","is_weekend",
    "lag_1","lag_7","rolling_3","rolling_7"
]

future_df["pred_count"] = model.predict(future_df[features])

future_df.head()


,x,y,weekday,is_weekend,t,lag_1,lag_7,rolling_3,rolling_7,pred_count
0,1,13,0,False,36,NaN,NaN,NaN,NaN,1.053685
1,1,18,0,False,36,NaN,NaN,NaN,NaN,1.050714
2,1,19,0,False,36,NaN,NaN,NaN,NaN,1.050714
3,1,20,0,False,36,NaN,NaN,NaN,NaN,1.050714
4,1,22,0,False,36,NaN,NaN,NaN,NaN,1.050714


In [13]:
# === 建立 circles_df（整合 HIGH/MED/LOW） ===

all_clusters = []

def extract_largest_cluster(df_level, level_name):
    valid = df_level[df_level["cluster"] != -1]

    if len(valid) == 0:
        print(f"No valid cluster for {level_name}")
        return

    largest_cluster = (
        valid.groupby("cluster")
        .size()
        .idxmax()
    )

    group = valid[valid["cluster"] == largest_cluster]

    cx = group["x"].mean()
    cy = group["y"].mean()
    r = np.sqrt(((group["x"] - cx)**2 + (group["y"] - cy)**2).max())

    all_clusters.append({
        "level": level_name,
        "center_x": cx,
        "center_y": cy,
        "radius_grid": r,
        "points": len(group)
    })

# 執行
extract_largest_cluster(high_df, "HIGH")
extract_largest_cluster(med_df, "MED")
extract_largest_cluster(low_df, "LOW")

circles_df = pd.DataFrame(all_clusters)

print("circles_df created.")
display(circles_df)

circles_df created.


,level,center_x,center_y,radius_grid,points
0,HIGH,25.000000,153.187500,3.941466,32
1,MED,26.041379,156.375862,24.389615,290
2,LOW,35.866972,143.981651,25.321233,218


In [14]:
# circles_df 可能是 list -> DataFrame
if not isinstance(circles_df, pd.DataFrame):
    circles_df = pd.DataFrame(circles_df)

print("circles_df columns:", circles_df.columns.tolist())
display(circles_df.head())

# ---- 欄位統一：以「grid 圈圈」為主 ----
rename_map = {}

# 常見變體：center x/y
if "cx" in circles_df.columns and "center_x" not in circles_df.columns:
    rename_map["cx"] = "center_x"
if "cy" in circles_df.columns and "center_y" not in circles_df.columns:
    rename_map["cy"] = "center_y"

# 半徑欄位：統一成 radius_grid（注意：此時還在 grid 單位）
for c in ["radius_grid", "radius", "r", "radius_cells"]:
    if c in circles_df.columns and "radius_grid" not in circles_df.columns:
        rename_map[c] = "radius_grid"
        break

# 點數欄位
for c in ["points", "n", "size", "n_points"]:
    if c in circles_df.columns and "points" not in circles_df.columns:
        rename_map[c] = "points"
        break

# 等級欄位
for c in ["level", "conf", "label", "tier"]:
    if c in circles_df.columns and "level" not in circles_df.columns:
        rename_map[c] = "level"
        break

if rename_map:
    circles_df = circles_df.rename(columns=rename_map)

required = {"level", "center_x", "center_y", "radius_grid", "points"}
missing = required - set(circles_df.columns)
if missing:
    raise KeyError(f"circles_df missing required columns: {missing}. Current: {circles_df.columns.tolist()}")

print("✓ circles_df schema OK")


circles_df columns: ['level', 'center_x', 'center_y', 'radius_grid', 'points']


,level,center_x,center_y,radius_grid,points
0,HIGH,25.000000,153.187500,3.941466,32
1,MED,26.041379,156.375862,24.389615,290
2,LOW,35.866972,143.981651,25.321233,218


✓ circles_df schema OK


In [15]:
from src.geo.grid_to_latlng import GridLatLngMapper

anchors = [
    {"x": 35, "y": 150, "lat": 43.068661, "lng": 141.350755},
    {"x": 20, "y": 120, "lat": 43.040000, "lng": 141.320000},
    {"x": 60, "y": 170, "lat": 43.100000, "lng": 141.400000},
]

mapper = GridLatLngMapper(anchors)

# 先轉換
circles_geo = mapper.transform(
    circles_df,
    x_col="center_x",
    y_col="center_y"
)

# 再印
print(circles_geo["level"].value_counts())

circles_geo.head()

level
HIGH    1
MED     1
LOW     1
Name: count, dtype: int64


,level,center_x,center_y,radius_grid,points,lat,lng
0,HIGH,25.000000,153.187500,3.941466,32,43.062252,141.331808
1,MED,26.041379,156.375862,24.389615,290,43.064847,141.334017
2,LOW,35.866972,143.981651,25.321233,218,43.066072,141.352012


In [16]:
# ===== grid 半徑 → 公尺 =====

GRID_SIZE_M = 250  # 依你們資料設定

circles_geo["radius_meter"] = (
    circles_geo["radius_grid"].astype(float) * GRID_SIZE_M
)

circles_geo.head()


,level,center_x,center_y,radius_grid,points,lat,lng,radius_meter
0,HIGH,25.000000,153.187500,3.941466,32,43.062252,141.331808,985.366564
1,MED,26.041379,156.375862,24.389615,290,43.064847,141.334017,6097.403799
2,LOW,35.866972,143.981651,25.321233,218,43.066072,141.352012,6330.308158


In [17]:
# ===== 清理小 cluster =====

MIN_POINTS = 15

circles_geo = circles_geo[
    circles_geo["points"] >= MIN_POINTS
].copy()

circles_geo.sort_values(["level","points"], ascending=False)


,level,center_x,center_y,radius_grid,points,lat,lng,radius_meter
1,MED,26.041379,156.375862,24.389615,290,43.064847,141.334017,6097.403799
2,LOW,35.866972,143.981651,25.321233,218,43.066072,141.352012,6330.308158
0,HIGH,25.000000,153.187500,3.941466,32,43.062252,141.331808,985.366564


In [18]:
# ===== 產生最終 API Output =====

level_order = {"HIGH": 3, "MED": 2, "LOW": 1}

circles_geo["level_rank"] = circles_geo["level"].map(level_order)

circles_geo = circles_geo.sort_values(
    ["level_rank", "points"],
    ascending=False
)

api_output = circles_geo[
    ["level", "lat", "lng", "radius_meter", "points"]
].to_dict(orient="records")

api_output[:5]

[{'level': 'HIGH',
  'lat': 43.06225217222231,
  'lng': 141.33180780555585,
  'radius_meter': 985.3665640892226,
  'points': 32},
 {'level': 'MED',
  'lat': 43.064847447969434,
  'lng': 141.33401717854437,
  'radius_meter': 6097.40379879932,
  'points': 290},
 {'level': 'LOW',
  'lat': 43.06607205270132,
  'lng': 141.35201231753314,
  'radius_meter': 6330.308157905319,
  'points': 218}]

In [19]:
import folium

m = folium.Map(location=[43.07, 141.35], zoom_start=12)

for c in api_output:
    color = {
        "HIGH": "red",
        "MED": "orange",
        "LOW": "blue"
    }[c["level"]]

    folium.Circle(
        location=[c["lat"], c["lng"]],
        radius=c["radius_meter"],
        color=color,
        fill=True,
        fill_opacity=0.3
    ).add_to(m)

from IPython.display import display
display(m)
